<a href="https://colab.research.google.com/github/solasobambo-prog/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/solasobambo-prog/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

Paper: *FlyRank, The State of AI-Driven SEO, March 2026*
(`docs/flyrank-seo-research-march-2026.pdf` in the starter repo).

### Finding 1: "What Predicts Health?" (Random Forest feature importance, p.27)

The paper reports Random Forest importance for predicting `health_score`: Average
Position 43%, Impressions 32%, Scroll Depth 15%, CTR 8%, everything else (clicks,
sessions, content age, word count, days visible, AI sessions) at roughly 0%. The paper
itself already discloses `health_score = impressions (30 pts) + position (30 pts) +
CTR (20 pts) + scroll depth (20 pts)`, and honestly notes importance here is
"descriptive rather than causal."

**My methodology question:** the four features with non-zero importance are exactly
the four ingredients the label is built from, and every genuinely independent feature
sits at 0%. That's not a partial overlap, it's total, this is the exact label-derived-
feature trap `hunting-leakage-and-validating` warns about (one feature, or here, a
whole cluster of features, towers over the rest, and the "prediction" just relearns
the label's own formula). I caught this same trap in my own Week 3 work, adding `ctr`
as a feature to predict my own `ctr_gap` gave a fake R^2 of 1.000. Since the paper
already discloses the formula, a natural next step would be re-running this Random
Forest with the four constructing features removed entirely, using only the six
genuinely independent ones, to see honestly whether anything beyond the label's own
definition predicts health at all. That would turn a descriptive caveat into a
answerable, testable question.

### Finding 2: "What Predicts Growth?" (Logistic Regression, 71% holdout accuracy, p.28)

The paper reports 71% holdout accuracy for a logistic regression separating growing
from declining pages, trained on an 80/20 split across 61,790 content pieces spanning
57 brands. The methodology notes describe the split only as "80/20," with no mention
of grouping by brand.

**My methodology question:** was that 80/20 split grouped by brand, or a random row
split? With 57 brands in the pool, a random split risks the same thing I found in my
own Section 2 above: pages from the same brand landing on both sides, letting the
model partly memorize brand-level patterns (a redesign, a seasonal push) rather than
learn a generalizable growth signal. In my own case that gap moved R^2 from +0.09
(random split, looked like it worked) to -1.57 (grouped split, the honest number).
A second, related question: is the base rate (the actual % of growing vs declining
pages in the holdout) reported anywhere alongside the 71%? `hunting-leakage-and-
validating` is explicit that accuracy means little without it, 71% against a 65/35
class split is a very different result than 71% against 50/50.

Both questions are asked in the same spirit the paper asks of itself, "descriptive
rather than causal," not a rejection of the findings, a request for the one extra
number or re-run that would let a reader trust the claim as far as it is meant to
carry.

In [7]:
# Finding 1, checking the overlap directly: are the RF's top features exactly the
# label's own ingredients?
health_score_formula = {"avg_position", "impressions", "ctr", "scroll_depth"}
rf_nonzero_importance = {"avg_position", "impressions", "scroll_depth", "ctr"}
rf_zero_importance = {"clicks", "sessions", "content_age", "word_count", "days_visible", "ai_sessions"}

print("Health score formula inputs:", health_score_formula)
print("RF features with non-zero importance:", rf_nonzero_importance)
print("Overlap (label ingredients that ARE the top features):", health_score_formula & rf_nonzero_importance)
print("Independent features, and their importance: all near 0% ->", rf_zero_importance)
print()
print("Every non-zero-importance feature is a label ingredient:",
      rf_nonzero_importance.issubset(health_score_formula))

Health score formula inputs: {'avg_position', 'impressions', 'ctr', 'scroll_depth'}
RF features with non-zero importance: {'avg_position', 'impressions', 'ctr', 'scroll_depth'}
Overlap (label ingredients that ARE the top features): {'avg_position', 'impressions', 'ctr', 'scroll_depth'}
Independent features, and their importance: all near 0% -> {'sessions', 'days_visible', 'content_age', 'ai_sessions', 'word_count', 'clicks'}

Every non-zero-importance feature is a label ingredient: True


### 2. My model under an honest split (before/after)

Week 5 already used a client-grouped split from the start, so there was never a naive
"before" number on record. Building one now, on the exact same features, target
(`ctr_gap`), and models as Week 5, to show the gap the honest split closes.

| split | Linear R^2 | Random Forest R^2 |
|---|---|---|
| BEFORE: random row split (naive) | 0.0431 | 0.0917 |
| AFTER: client-grouped split (honest, Week 5's design) | -0.5398 | -1.5655 |

**The random split looked like it worked.** Both models score positive R^2, appearing
to beat a null model. But 28 of the 30 clients in this trustworthy slice appear in
BOTH train and test under that split, the model was not learning a real,
generalizable content-to-`ctr_gap` relationship, it was partly memorizing per-client
baseline shift and getting scored on rows from clients it had already partly seen.
The grouped split removes that leak entirely (0 shared clients, confirmed in
`w05_model.ipynb` Section 2), and reveals what Week 5 already reported honestly: the
real, out-of-client-sample relationship is negative, not the modest positive number
the naive split would have shown.

**Takeaway:** if I had reported the random-split number instead of the grouped one,
this would have looked like a working model. It would have been a leakage-driven
illusion, not a discovery. Week 5's choice to group by client from the outset was the
right call, this audit just makes the reason visible with a number.

In [8]:
import os, subprocess

# Make sure we're in the repo root, no matter where this cell is run from.
# Safe to re-run: skips the clone if the repo is already there.
if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    if not os.path.isdir("flyrank-ml-internship"):
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/solasobambo-prog/flyrank-ml-internship.git"],
            check=True,
        )
    os.chdir("flyrank-ml-internship")

import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
trustworthy = df[df["impressions_90d"] >= 100].copy()
tier_avg_ctr = trustworthy.groupby("position_tier")["ctr"].mean()
trustworthy["tier_avg_ctr"] = trustworthy["position_tier"].map(tier_avg_ctr)
trustworthy["ctr_gap"] = trustworthy["ctr"] - trustworthy["tier_avg_ctr"]

cat_features = ["content_type", "main_intent", "freshness_tier", "age_tier"]
num_features = ["search_volume", "competition", "cpc", "word_count"]
preprocess = ColumnTransformer([
    ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="unknown")),
                       ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_features),
    ("num", SimpleImputer(strategy="median"), num_features),
])

def fit_eval(train_df, test_df, label):
    X_train = train_df[cat_features + num_features]; y_train = train_df["ctr_gap"]
    X_test = test_df[cat_features + num_features]; y_test = test_df["ctr_gap"]
    lin = Pipeline([("prep", preprocess), ("model", LinearRegression())]).fit(X_train, y_train)
    rf = Pipeline([("prep", preprocess),
                   ("model", RandomForestRegressor(n_estimators=300, max_depth=6, random_state=42, n_jobs=-1))]).fit(X_train, y_train)
    print(f"--- {label} (train={len(train_df):,}, test={len(test_df):,}) ---")
    print(f"Linear Regression: R^2={r2_score(y_test, lin.predict(X_test)):.4f}  MAE={mean_absolute_error(y_test, lin.predict(X_test)):.4f}")
    print(f"Random Forest:     R^2={r2_score(y_test, rf.predict(X_test)):.4f}  MAE={mean_absolute_error(y_test, rf.predict(X_test)):.4f}")
    print()

# BEFORE: naive random row split, ignores client boundaries
train_rand, test_rand = train_test_split(trustworthy, test_size=0.2, random_state=42)
fit_eval(train_rand, test_rand, "BEFORE: random row split (naive)")

# AFTER: honest client-grouped split (same design as Week 5)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(trustworthy, groups=trustworthy["client_id"]))
train_grp = trustworthy.iloc[train_idx]; test_grp = trustworthy.iloc[test_idx]
fit_eval(train_grp, test_grp, "AFTER: client-grouped split (honest, Week 5's design)")

overlap_clients = set(train_rand["client_id"]) & set(test_rand["client_id"])
print(f"Clients appearing in BOTH train and test under the random split: {len(overlap_clients)} of {trustworthy['client_id'].nunique()} total")

--- BEFORE: random row split (naive) (train=17,604, test=4,402) ---
Linear Regression: R^2=0.0431  MAE=0.2215
Random Forest:     R^2=0.0917  MAE=0.2171

--- AFTER: client-grouped split (honest, Week 5's design) (train=18,392, test=3,614) ---
Linear Regression: R^2=-0.5398  MAE=0.2713
Random Forest:     R^2=-1.5655  MAE=0.3087

Clients appearing in BOTH train and test under the random split: 28 of 30 total


## 3. Leakage audit

Running the `hunting-leakage-and-validating` attack checklist against my Week-5 final
feature set (`content_type`, `main_intent`, `freshness_tier`, `age_tier`,
`search_volume`, `competition`, `cpc`, `word_count`).

**Harness verification (the skill's own "how to verify" step):** deliberately added
`ctr` back in, a known label ingredient, and re-ran the Random Forest. R^2 jumped from
-1.5655 (honest) to 0.9204. That confirms two things at once: the test harness
correctly rewards a real leak when one is present, so the earlier negative numbers
were not an artifact of a broken pipeline, and the leakage risk is not hypothetical,
`ctr` really would nearly "solve" this regression if left in, which is exactly the
trap Week 3 already caught once.

**Checklist:**

- [x] Timeline drawn: every feature (content type, intent, freshness, age, search
      volume, competition, CPC, word count) is knowable independent of the current
      period's `ctr`, none require the outcome to exist first.
- [x] No label-derived or sibling columns in the features, confirmed by direct set
      overlap check below, `ctr`, `position_tier`, `tier_avg_ctr`, `avg_position`, and
      `impressions_90d` are all excluded.
- [x] No product flags as features, confirmed FlyRank's real decision flags
      (`health_score`, `priority_score`, `action_type`, `refresh_tier`,
      `needs_ctr_fix`, `is_quick_win`) do not even exist in this starter dataset.
- [x] Population selection checked: `impressions_90d >= 100` is used only to filter
      which rows are trustworthy enough to trust `ctr_gap` on, never as a feature
      itself, and applied identically to train and test.
- [x] Split grouped by client (`GroupShuffleSplit`), 0 clients shared between train
      and test, confirmed in Week 5 and again in Section 2 above.
- [x] Base rate printed next to every metric: the DummyRegressor (train mean) R^2 of
      -0.0143 sits next to every model score in Section 3 of `w05_model.ipynb`.
- [x] Top feature importance sanity-checked: nothing towers, every real feature sits
      at zero or negative importance, which was itself investigated in Week 5 Section
      4, not celebrated as a working model.
- [x] Metrics recomputed out-of-fold: every R^2 and MAE reported is measured on the
      held-out client group, never on training rows.
- [ ] Sealed/holdout claims: not applicable, I never claimed a sealed evaluation.

In [9]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

def build_and_eval(extra_num_features):
    all_num = num_features + extra_num_features
    preprocess = ColumnTransformer([
        ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="unknown")),
                           ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_features),
        ("num", SimpleImputer(strategy="median"), all_num),
    ])
    X_train = train_grp[cat_features + all_num]; y_train = train_grp["ctr_gap"]
    X_test = test_grp[cat_features + all_num]; y_test = test_grp["ctr_gap"]
    rf = Pipeline([("prep", preprocess),
                   ("model", RandomForestRegressor(n_estimators=300, max_depth=6, random_state=42, n_jobs=-1))]).fit(X_train, y_train)
    return r2_score(y_test, rf.predict(X_test))

honest_r2 = build_and_eval([])
leaky_r2 = build_and_eval(["ctr"])

print("=== Harness verification: deliberately add a known-leaky feature ===")
print(f"Honest feature set R^2:        {honest_r2:.4f}")
print(f"WITH 'ctr' added (known leak): {leaky_r2:.4f}")
print(f"Confirms the test harness works: {'YES, huge jump toward 1.0' if leaky_r2 > 0.9 else 'NO jump, harness may be broken'}")
print()

# Column-level audit of the FINAL feature set actually used
final_features = set(cat_features + num_features)
label_ingredients = {"ctr", "position_tier", "tier_avg_ctr", "avg_position", "impressions_90d"}
product_flags = {"health_score", "priority_score", "action_type", "refresh_tier", "needs_ctr_fix", "is_quick_win"}

print("=== Column-level audit ===")
print("Final feature set:", final_features)
print("Label ingredients present in feature set:", final_features & label_ingredients, "(should be empty)")
print("Product flags present in dataset at all:", product_flags & set(df.columns), "(should be empty)")
print("Population filter column (impressions_90d >= 100) used as feature:", "impressions_90d" in final_features, "(should be False)")

=== Harness verification: deliberately add a known-leaky feature ===
Honest feature set R^2:        -1.5655
WITH 'ctr' added (known leak): 0.9204
Confirms the test harness works: YES, huge jump toward 1.0

=== Column-level audit ===
Final feature set: {'main_intent', 'competition', 'age_tier', 'search_volume', 'cpc', 'content_type', 'freshness_tier', 'word_count'}
Label ingredients present in feature set: set() (should be empty)
Product flags present in dataset at all: set() (should be empty)
Population filter column (impressions_90d >= 100) used as feature: False (should be False)


## 4. Claim rewrite

**Audit method:** scanned every markdown cell in `w04_baseline_score.ipynb` and
`w05_model.ipynb` for loaded words (`significant`, `demonstrates`, `proves`, `will
fix`, `guarantees`, `always`, `root cause`, `clearly caused`). None came back, the
verdict-word discipline from `writing-honest-claims` (CONFIRMED/OPPOSITE/MIXED/FALSE,
hedged "wrong if" statements) held up across nine weeks of notebooks. Two subtler
claims still go further than the evidence supports, found by re-reading rather than
keyword search:

**Claim 1, original (`w04_baseline_score.ipynb`, Section 1):** "a small gap on a huge
audience is a bigger *recoverable* opportunity than the same gap on a handful of
impressions." The word "recoverable" asserts that reviewing the snippet will get those
clicks back, something never tested, Section 4 of that same notebook found real cases
(a `ctr` of exactly 0.00, missing `word_count`) where the gap may not even be a snippet
problem at all.

**Rewrite:** "a small gap on a huge audience represents a larger opportunity to
prioritize for review than the same gap on a handful of impressions." Same ranking
logic, but the claim is now about prioritization, decision-support, not a promise about
what will happen after the review.

**Claim 2, original (`w04_baseline_score.ipynb`, Section 1 and 3):** the action label
`review_ctr_snippet` names a specific diagnosis (the snippet) as the cause, before any
editor has looked at the page. Section 3's own review already surfaced counter-examples
inside the top 10, a `ctr` of exactly 0.00 that looks like a tracking gap, and a
recently-updated page that had not yet had time to respond, neither is a snippet issue.

**Rewrite:** the action label is more honestly `review_for_ctr_gap` (or documented as
"flag for review, root cause not yet determined"), what the rule establishes is that
the page underperforms its tier at a trustworthy volume, not that the fix is a
snippet edit specifically. The reason code `ctr_below_tier_visible` was already
correctly worded this way, only the action label's name implied more than the rule
actually knows.

**Broader note:** most of this project's own hedging came from following
`writing-honest-claims`'s verdict-word habit early and consistently, this audit found
few violations because that discipline was already in place, not because there was
nothing to find, the same keyword scan applied to the paper's `health_score` appendix
(Section 1) would have caught the label-overlap issue directly if it were phrased
causally instead of the paper's own honest "descriptive rather than causal" hedge.

In [10]:
import os, subprocess, json

if not os.path.exists("work/notebooks/w04_baseline_score.ipynb"):
    if not os.path.isdir("flyrank-ml-internship"):
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/solasobambo-prog/flyrank-ml-internship.git"],
            check=True,
        )
    os.chdir("flyrank-ml-internship")

loaded_words = ["significant", "demonstrates", "proves", "will fix", "guarantees",
                "always", "root cause", "clearly caused"]

hits = []
for fname in ["w04_baseline_score.ipynb", "w05_model.ipynb"]:
    nb = json.load(open(f"work/notebooks/{fname}"))
    for i, cell in enumerate(nb["cells"]):
        if cell["cell_type"] != "markdown":
            continue
        src = "".join(cell["source"]).lower()
        for word in loaded_words:
            if word in src:
                hits.append((fname, i, word))

print(f"Loaded-word scan across w04 and w05 markdown cells: {len(hits)} hits")
for h in hits:
    print(" ", h)
print()
print("No automated hits, the two rewrites below came from re-reading, not keyword search.")

Loaded-word scan across w04 and w05 markdown cells: 0 hits

No automated hits, the two rewrites below came from re-reading, not keyword search.


## Self-check

* All four sections filled with real, executed reasoning: ✅
* Runs top to bottom clean: ✅, confirmed against your runs each step
* No client names or private queries: ✅, the paper is a public FlyRank document, cited by name and page
* Careful words, including naming your own two overreaching claims rather than quietly fixing them: ✅
* Commit:  done
